In [1]:
from the_well.data.datasets import WellDataset
from pathlib import Path
import sys
import os

from pathlib import Path as _Path
PROJECT_ROOT = str(next(p for p in [_Path.cwd(), *_Path.cwd().parents] if (p / "pyproject.toml").exists()))  # repo root, works on laptop and cluster
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.mini_well import create_mini_well

In [2]:
#well_base_path    = '/Users/gabrieleaccarino/Workspace/multi-scale-wavelet-flow-matching/data/datasets'
well_base_path    = '/glade/work/gaccarino/data/datasets'
#well_dataset_name = 'turbulent_radiative_layer_2D'
#well_dataset_name = 'rayleigh_benard'
#well_dataset_name = 'planetswe'
#well_dataset_name = 'active_matter'
well_dataset_name = "viscoelastic_instability"

MINI_DATASET  = f'{well_dataset_name}_mini'

for split in ['train', 'valid', 'test']:
    ds = WellDataset(
        well_base_path=well_base_path,
        well_dataset_name=well_dataset_name,
        well_split_name=split,
    )
    print(f"\n── {split} ──")
    print(f"  Spatial resolution   : {ds.size_tuple}")
    print(f"  N files              : {ds.n_files}")
    print(f"  Trajectories/file    : {ds.n_trajectories_per_file}")
    print(f"  Steps/traj           : {ds.n_steps_per_trajectory}")
    print(f"  Total trajectories   : {sum(ds.n_trajectories_per_file)}")


── train ──
  Spatial resolution   : (512, 512)
  N files              : 7
  Trajectories/file    : [16, 33, 28, 35, 30, 31, 40]
  Steps/traj           : [20, 60, 60, 20, 20, 20, 20]
  Total trajectories   : 213

── valid ──
  Spatial resolution   : (512, 512)
  N files              : 7
  Trajectories/file    : [2, 3, 3, 4, 3, 3, 4]
  Steps/traj           : [20, 60, 60, 20, 20, 20, 20]
  Total trajectories   : 22

── test ──
  Spatial resolution   : (512, 512)
  N files              : 7
  Trajectories/file    : [2, 3, 3, 4, 3, 3, 4]
  Steps/traj           : [20, 60, 60, 20, 20, 20, 20]
  Total trajectories   : 22


# Rayleigh-Bénard

In [ ]:
# ── Resampling config ─────────────────────────────────────────
SPATIAL_FACTOR = 1    # no spatial downsampling
TIME_FRACTION  = 1.0  # use all steps per trajectory
TIME_FACTOR    = 1    # no temporal downsampling

MAX_TRAJ_PER_FILE = {'train': 10, 'valid': 5, 'test': 5}

In [ ]:
# ── Parameter subset ─────────────────────────────────────────
ds_full = WellDataset(
    well_base_path=well_base_path,
    well_dataset_name=well_dataset_name,
    well_split_name='train',  # names are identical across splits
)
all_files = sorted(ds_full.files_paths)

KEEP_RA = ['1e6', '1e8', '1e10']   # soft / hard / ultimate turbulence
KEEP_PR = ['1e-1', '1', '10']      # low / mid / high Prandtl

# Anchor Pr match to .hdf5 suffix to avoid '1' matching '10' or '1e-1'
selected_names = [
    Path(f).name for f in all_files
    if any(f'Rayleigh_{ra}_' in Path(f).name for ra in KEEP_RA)
    and any(f'Prandtl_{pr}.hdf5' in Path(f).name for pr in KEEP_PR)
]
print(f"Keeping {len(selected_names)} files:")
for name in selected_names:
    print(f"  {name}")

In [ ]:
# ── Build mini dataset ────────────────────────────────────────
for split, max_traj in MAX_TRAJ_PER_FILE.items():
    print(f"\n{'='*50}  {split}  {'='*50}")
    for fname in selected_names:
        ds_single = WellDataset(
            well_base_path=well_base_path,
            well_dataset_name=well_dataset_name,
            well_split_name=split,
            include_filters=[fname],
        )
        create_mini_well(
            dataset=ds_single,
            well_base_path=well_base_path,
            mini_dataset_name=MINI_DATASET,
            spatial_downsample_factor=SPATIAL_FACTOR,
            time_downsample_factor=TIME_FACTOR,
            max_trajectories=max_traj,
            split=split,
            time_fraction=TIME_FRACTION,
        )

print("\nAll splits done ✓")

# PlanetSWE

In [ ]:
# ── Resampling config ─────────────────────────────────────────
SPATIAL_FACTOR = 1    # no spatial downsampling
TIME_FRACTION  = 1.0  # 1008 × 1.0 ~ 1008 steps, this is 1 model year 
TIME_FACTOR    = 6    # downsample every 6 steps → 168 effective steps
                      # each retained step = 6 sim-hours
                      # daily  cycle: 24/6 =  4 points per day 
                      # annual cycle: 1008/6 = 168 points/year

MAX_TRAJ_PER_FILE = {'train': 1, 'valid': 1, 'test': 1}

In [ ]:
# ── Parameter subset ─────────────────────────────────────────
ds_splits = {
    split: WellDataset(
        well_base_path=well_base_path,
        well_dataset_name=well_dataset_name,
        well_split_name=split,
    )
    for split in ['train', 'valid', 'test']
}

# ── File selection: ALL ICs, only s1 (year 1 out of 3 years) ─────────────────
#   train: IC00–IC31, s1 only → 32 files
#   valid: IC32–IC35, s1 only →  4 files
#   test:  IC36–IC39, s1 only →  4 files

file_selection = {
    split: [
        Path(f).name
        for f in sorted(ds_splits[split].files_paths)
        if Path(f).stem.endswith('_s1')      # first model year only
    ]
    for split in ['train', 'valid', 'test']
}


#KEEP_ICS = [f'IC{i:02d}' for i in range(0, 32, 4)]
#print(f"Keeping train ICs: {KEEP_ICS}")

#file_selection = {
#    'train': [Path(f).name for f in sorted(ds_splits['train'].files_paths) if any(ic in Path(f).name for ic in KEEP_ICS)],
#    'valid': [Path(f).name for f in sorted(ds_splits['valid'].files_paths)],
#    'test' : [Path(f).name for f in sorted(ds_splits['test'].files_paths)],
#}

for split, fnames in file_selection.items():
    print(f"{split}: {len(fnames)} files")

In [ ]:
for split in ['train', 'valid', 'test']:
    print(f"\n{'='*50}  {split}  {'='*50}")
    for fname in file_selection[split]:
        ds_single = WellDataset(
            well_base_path=well_base_path,
            well_dataset_name=well_dataset_name,
            well_split_name=split,
            include_filters=[fname],
        )
        create_mini_well(
            dataset=ds_single,
            well_base_path=well_base_path,
            mini_dataset_name=MINI_DATASET,
            spatial_downsample_factor=SPATIAL_FACTOR,
            time_downsample_factor=TIME_FACTOR,
            max_trajectories=1,
            split=split,
            time_fraction=TIME_FRACTION,
        )

print("\nAll splits done ✓")

# Active Matter

In [ ]:
# ── Resampling config ─────────────────────────────────────────
SPATIAL_FACTOR    = 1    # no spatial downsampling (256×256 kept as-is)
TIME_FRACTION     = 1.0  # use all 81 time-steps per trajectory
TIME_FACTOR       = 1    # no temporal downsampling
MAX_TRAJ_PER_FILE = {'train': 5, 'valid': 5, 'test': 5}  # 5 traj per file max

# ── Parameter subset ──────────────────────────────────────────
# ζ ∈ {1.0, 5.0, 11.0, 17.0} applied to ALL splits
#   → isotropic / pre-transition / near-transition / deep nematic
# α ∈ {-1.0, -2.0, -3.0, -4.0, -5.0} applied to TRAIN only (to control train size)
#   → valid/test keep all α present in their split for that ζ
#
# Result:
#   train : 4ζ × 5α = 20 files  (selected α)
#   valid :           7 files   (all α available for selected ζ)
#   test  :          11 files   (all α available for selected ζ)

KEEP_ZETA        = ['1.0', '5.0', '11.0', '17.0']
KEEP_ALPHA_TRAIN = ['-1.0', '-2.0', '-3.0', '-4.0', '-5.0']

def _zeta_match(fname: str) -> bool:
    """Keep only files whose ζ is in KEEP_ZETA."""
    return any(f'zeta_{z}_' in fname for z in KEEP_ZETA)

def _alpha_match(fname: str) -> bool:
    """Keep only files whose α is in KEEP_ALPHA_TRAIN (train only)."""
    return any(fname.endswith(f'alpha_{a}.hdf5') for a in KEEP_ALPHA_TRAIN)

# ── Build per-split dataset objects (splits DO have physical files) ────────────
ds_splits = {
    split: WellDataset(
        well_base_path=well_base_path,
        well_dataset_name=well_dataset_name,
        well_split_name=split,
    )
    for split in ['train', 'valid', 'test']
}

# ── File selection: ζ filter for all splits, +α filter for train ──────────────
file_selection = {
    'train': [
        Path(f).name for f in sorted(ds_splits['train'].files_paths)
        if _zeta_match(Path(f).name) and _alpha_match(Path(f).name)
    ],
    'valid': [
        Path(f).name for f in sorted(ds_splits['valid'].files_paths)
        if _zeta_match(Path(f).name)
    ],
    'test': [
        Path(f).name for f in sorted(ds_splits['test'].files_paths)
        if _zeta_match(Path(f).name)
    ],
}

for split, fnames in file_selection.items():
    print(f"{split}: {len(fnames)} files")
    for name in fnames:
        print(f"  {name}")

# Sanity: all splits must be non-empty
for split, fnames in file_selection.items():
    assert len(fnames) > 0, f"{split} split is empty — check KEEP_ZETA / KEEP_ALPHA_TRAIN"

# ── Build mini dataset ────────────────────────────────────────
for split, max_traj in MAX_TRAJ_PER_FILE.items():
    print(f"\n{'='*50}  {split}  {'='*50}")
    for fname in file_selection[split]:
        ds_single = WellDataset(
            well_base_path=well_base_path,
            well_dataset_name=well_dataset_name,
            well_split_name=split,
            include_filters=[fname],
        )
        create_mini_well(
            dataset=ds_single,
            well_base_path=well_base_path,
            mini_dataset_name=MINI_DATASET,
            spatial_downsample_factor=SPATIAL_FACTOR,
            time_downsample_factor=TIME_FACTOR,
            max_trajectories=max_traj,
            split=split,
            time_fraction=TIME_FRACTION,
        )

print("\nAll splits done ✓")

# Viscoelastic Instability

── train ──
  Spatial resolution   : (512, 512)
  N files              : 7
  Trajectories/file    : [16, 33, 28, 35, 30, 31, 40]
  Steps/traj           : [20, 60, 60, 20, 20, 20, 20]
  Total trajectories   : 213

── valid ──
  Spatial resolution   : (512, 512)
  N files              : 7
  Trajectories/file    : [2, 3, 3, 4, 3, 3, 4]
  Steps/traj           : [20, 60, 60, 20, 20, 20, 20]
  Total trajectories   : 22

── test ──
  Spatial resolution   : (512, 512)
  N files              : 7
  Trajectories/file    : [2, 3, 3, 4, 3, 3, 4]
  Steps/traj           : [20, 60, 60, 20, 20, 20, 20]
  Total trajectories   : 22

In [4]:
# ── Resampling config ─────────────────────────────────────────
SPATIAL_FACTOR    = 1
TIME_FACTOR       = 1
MAX_TRAJ_PER_FILE = {'train': 16, 'valid': 2, 'test': 2}

# Files with 60 steps get fraction 20/60 = 1/3; all others (20 steps) keep 1.0
TIME_FRACTION_MAP = {
    'viscoelastic_instability_CAR.hdf5': 20 / 60,
    'viscoelastic_instability_EIT.hdf5': 20 / 60,
}
DEFAULT_TIME_FRACTION = 1.0   # already 20 steps

# ── Discover all files from the train split ───────────────────
ds_full = WellDataset(
    well_base_path=well_base_path,
    well_dataset_name=well_dataset_name,
    well_split_name='train',
)
all_file_names = sorted(Path(f).name for f in ds_full.files_paths)
print(f"Found {len(all_file_names)} files:")
for name in all_file_names:
    steps = 60 if name in TIME_FRACTION_MAP else 20
    print(f"  {name}  ({steps} steps → capped to 20)")

# ── Build mini dataset ────────────────────────────────────────
for split, max_traj in MAX_TRAJ_PER_FILE.items():
    print(f"\n{'='*50}  {split}  {'='*50}")
    for fname in all_file_names:
        time_fraction = TIME_FRACTION_MAP.get(fname, DEFAULT_TIME_FRACTION)
        ds_single = WellDataset(
            well_base_path=well_base_path,
            well_dataset_name=well_dataset_name,
            well_split_name=split,
            include_filters=[fname],
        )
        create_mini_well(
            dataset=ds_single,
            well_base_path=well_base_path,
            mini_dataset_name=MINI_DATASET,
            spatial_downsample_factor=SPATIAL_FACTOR,
            time_downsample_factor=TIME_FACTOR,
            max_trajectories=max_traj,
            split=split,
            #time_fraction=time_fraction,
            time_fraction=1.0,  # keep all steps, even if some files have 60 steps instead of 20 (no capping
        )

print("\nAll splits done ✓")

# Double check

In [3]:
# Cell 5 — Verify the mini dataset
for split in ['train', 'valid', 'test']:
    ds_mini = WellDataset(
        path=os.path.join(well_base_path, MINI_DATASET, 'data', split), 
        n_steps_input=4,
        n_steps_output=1,
        #well_base_path=well_base_path,
        #well_dataset_name=MINI_DATASET,
        #well_split_name=split,
    )
    total_traj = sum(ds_mini.n_trajectories_per_file)
    print(f"\n── mini/{split} ──")
    print(f"  Spatial resolution : {ds_mini.size_tuple}")
    print(f"  N files            : {ds_mini.n_files}")
    print(f"  Trajectories/file  : {ds_mini.n_trajectories_per_file}")
    print(f"  Steps/traj         : {ds_mini.n_steps_per_trajectory}")
    print(f"  Total trajectories : {total_traj}")
    print(f"  Total windows      : {len(ds_mini)}")


── mini/train ──
  Spatial resolution : (512, 512)
  N files            : 7
  Trajectories/file  : [16, 16, 16, 16, 16, 16, 16]
  Steps/traj         : [20, 60, 60, 20, 20, 20, 20]
  Total trajectories : 112
  Total windows      : 3072

── mini/valid ──
  Spatial resolution : (512, 512)
  N files            : 7
  Trajectories/file  : [2, 2, 2, 2, 2, 2, 2]
  Steps/traj         : [20, 60, 60, 20, 20, 20, 20]
  Total trajectories : 14
  Total windows      : 384

── mini/test ──
  Spatial resolution : (512, 512)
  N files            : 7
  Trajectories/file  : [2, 2, 2, 2, 2, 2, 2]
  Steps/traj         : [20, 60, 60, 20, 20, 20, 20]
  Total trajectories : 14
  Total windows      : 384
